# 实验二：图像读取、颜色转换、噪声与滤波

依据 [CV_exp / 2nd.exp](https://github.com/Guyudong1/CV_exp/tree/main/2nd.exp) 的五步实验，用用户提供的图片完成实验。默认原图为 `assets/input.jpg`，在完整分辨率上计算，不缩小原图。

实验流程：读取图像及像素 → BGR/RGB/灰度转换 → 添加椒盐及高斯噪声 → 均值/中值/高斯滤波对比 → 手动实现彩色中值滤波。算法和输出统一由同目录的 `experiment.py` 提供；本 Notebook 展示参数、结果和实时指标。

先在仓库目录创建 Python 3.12 虚拟环境，按实验报告安装依赖：`.venv\Scripts\python.exe -m pip install -r exp2/requirements.txt`。随后选用安装了这些依赖的 Python 内核，从头运行全部单元格。允许从仓库根目录或 `exp2` 目录启动；完整执行会重新生成 `result` 下的结果文件。

In [ ]:
from pathlib import Path
import csv
import inspect
import sys

import cv2
from IPython.display import Image, Markdown, display

root = Path.cwd()
exp_dir = root / 'exp2' if (root / 'exp2' / 'experiment.py').exists() else root
assert (exp_dir / 'experiment.py').is_file(), '请从仓库根目录或 exp2 目录启动。'
if str(exp_dir) not in sys.path:
    sys.path.insert(0, str(exp_dir))
from experiment import read_image, run_experiment, manual_median_filter_color

input_path = exp_dir / 'assets' / 'input.jpg'
output_dir = exp_dir / 'result'
SEED = 2026
KERNEL_SIZE = 5
assert input_path.is_file(), f'原图不存在：{input_path}'
print('输入：', input_path)
print('结果目录：', output_dir)

## 1. 读取图像并观察像素

`read_image` 使用 OpenCV 解码，返回 `uint8` 的 BGR 三通道图像，并兼容中文路径。数组索引 `[100, 100]` 表示第 100 行、第 100 列（从 0 开始）。

下面仅调用一次 `run_experiment`，统一生成后续章节使用的图像、`metrics.csv` 和 `summary.json`。随机数种子固定，便于复现。

In [ ]:
bgr = read_image(input_path)
summary = run_experiment(input_path, output_dir, seed=SEED, kernel_size=KERNEL_SIZE)
info = summary['input']
print(f"图像：宽 {info['width']} × 高 {info['height']}，{info['channels']} 通道，{info['dtype']}")
print('OpenCV 数组形状 (H, W, C)：', bgr.shape)
print('[100, 100] BGR：', info['pixel_bgr'])
print('[100, 100] RGB：', info['pixel_rgb'])
print('[100, 100] 灰度：', info['pixel_gray'])
display(Image(filename=str(input_path), width=450))

## 2. BGR、RGB 与灰度转换

Matplotlib 按 RGB 解释三通道数组，直接显示 OpenCV 的 BGR 数组会交换红、蓝颜色。`experiment.py` 使用以下转换：

```python
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
```

灰度转换按通道权重合成亮度，不等于三个通道的简单平均。对比图依次为 BGR 数组直接显示的错误颜色、正确 RGB、灰度。

In [ ]:
print('BGR → RGB → BGR 无损还原：', summary['validation']['bgr_rgb_roundtrip'])
display(Image(filename=str(output_dir / 'color_spaces.png'), width=1100))

## 3. 添加椒盐噪声与高斯噪声

采用参考实验的参数。`skimage.util.random_noise` 将 `uint8` 输入转为 `[0, 1]` 浮点数：椒盐噪声 `amount=0.4`，盐、椒比例各半；高斯噪声 `mean=0.2`、`var=0.03`。两类噪声分别使用种子 2026、2027 的随机数流。

```python
sp_float = random_noise(rgb, mode='s&p', amount=0.4,
                        rng=np.random.default_rng(SEED))
gaussian_float = random_noise(rgb, mode='gaussian', mean=0.2, var=0.03,
                              rng=np.random.default_rng(SEED + 1))
```

本实现将浮点结果裁剪至 `[0, 1]`，乘以 255 并四舍五入转成 `uint8`，供全部 OpenCV 滤波器使用。`amount` 是随机修改通道元素的概率；多通道图像实际受影响的像素比例以及未改变的原有极值，应以实测为准。高斯噪声均值为正，且超出范围的数值会被裁剪。

In [ ]:
params = summary['parameters']
observations = summary['noise_observations']
print('椒盐 amount：', params['salt_pepper_amount'])
print('高斯 mean / variance：', params['gaussian_mean'], '/', params['gaussian_variance'])
print(f"椒盐实际改变的通道值比例：{observations['salt_pepper_changed_channel_fraction']:.2%}")
print(f"椒盐实际改变的像素比例：{observations['salt_pepper_changed_pixel_fraction']:.2%}")
print(f"高斯上限裁剪比例：{observations['gaussian_upper_clipped_channel_fraction']:.2%}")
print(f"高斯下限裁剪比例：{observations['gaussian_lower_clipped_channel_fraction']:.2%}")
display(Image(filename=str(output_dir / 'noise_comparison.png'), width=1100))

## 4. 均值、中值和高斯滤波对比

对每类噪声的同一份 `uint8` 输入，分别使用 5×5 窗口。边界均采用复制边缘像素；高斯滤波的 `sigmaX=0`，由 OpenCV 根据窗口大小确定标准差。

```python
mean_result = cv2.blur(noisy, (5, 5), borderType=cv2.BORDER_REPLICATE)
median_result = cv2.medianBlur(noisy, 5)
gaussian_result = cv2.GaussianBlur(noisy, (5, 5), 0,
                                   borderType=cv2.BORDER_REPLICATE)
```

2×3 对比图：第一行为椒盐噪声，第二行为高斯噪声；每行依次为均值、中值、高斯滤波。MSE 越小越好，PSNR 和 SSIM 越大越好；平均误差为输出减原图的通道值平均，用于观察亮度偏差。所有指标都对照原始 RGB 图像，在完整分辨率上计算。

In [ ]:
display(Image(filename=str(output_dir / 'filter_results_2x3.png'), width=1100))
with (output_dir / 'metrics.csv').open(encoding='utf-8', newline='') as handle:
    metric_rows = list(csv.DictReader(handle))
noise_labels = {'salt_pepper': '椒盐', 'gaussian': '高斯'}
method_labels = {'noisy': '加噪原图', 'mean': '均值', 'median': '中值', 'gaussian': '高斯滤波'}
table = ['| 噪声 | 方法 | MSE | PSNR / dB | SSIM | 平均误差 |',
         '| --- | --- | ---: | ---: | ---: | ---: |']
for row in metric_rows:
    table.append(f"| {noise_labels[row['noise']]} | {method_labels[row['method']]} | "
                 f"{float(row['mse']):.2f} | {float(row['psnr_db']):.2f} | "
                 f"{float(row['ssim']):.4f} | {float(row['mean_error']):+.2f} |")
display(Markdown('\n'.join(table)))

## 5. 手动实现彩色中值滤波

对椒盐噪声图像的每个颜色通道，复制边界后取每个 5×5 邻域的中值。实现使用 NumPy 构造滑动窗口、分块处理连续 64 行，以控制完整图片的内存占用；不调用 OpenCV、SciPy 等库的现成滤波函数。逐通道处理不会把不同颜色通道混在同一个中值中。

下面展示公共函数的完整源码和本次运行的一致性结果。手动实现与 OpenCV 使用相同输入、窗口及边界；最大绝对差为 0 才能说明所有通道值完全相同。

In [ ]:
display(Markdown('```python\n' + inspect.getsource(manual_median_filter_color) + '\n```'))
print('手动中值与 OpenCV 完全一致：', summary['validation']['manual_matches_opencv'])
print('最大绝对差：', summary['validation']['manual_max_abs_difference'])
print('不同的通道值数量：', summary['validation']['manual_different_channel_values'])
display(Image(filename=str(output_dir / 'manual_median.png'), width=1100))

## 实测分析与结论

以下结论由当前运行的 `summary` 计算，分别报告 PSNR 和 SSIM 的最佳方法，并与未滤波的加噪图对照。不预先假定高斯滤波一定最优；不同指标和图像区域可能得出不同排序。视觉上还应检查发丝、眼睛边缘、花朵细节是否被平滑。

椒盐噪声是脉冲型干扰，中值滤波利用邻域排序排除极端值，但高比例噪声和较大窗口仍可能损失细节。均值及高斯滤波会平均邻域内容，也可能将脉冲扩散到邻近像素。

本实验的高斯噪声 `mean=0.2`，在裁剪前相当于平均增加 51 个灰度级；原图本身包含大量高亮区域，饱和裁剪改变了噪声分布并丢失部分信息。空间滤波可以减弱随机变化，不能自动消除这一系统性的偏亮。应结合平均误差和裁剪比例解释指标，不能把结论推广为所有零均值高斯噪声场景的结论。

In [ ]:
records = summary['metrics']
for noise in ('salt_pepper', 'gaussian'):
    group = [r for r in records if r['noise'] == noise]
    baseline = next(r for r in group if r['method'] == 'noisy')
    filtered = [r for r in group if r['method'] != 'noisy']
    best_psnr = max(filtered, key=lambda r: r['psnr_db'])
    best_ssim = max(filtered, key=lambda r: r['ssim'])
    print(f"{noise_labels[noise]}噪声：")
    print(f"  未滤波：PSNR {baseline['psnr_db']:.2f} dB，SSIM {baseline['ssim']:.4f}。")
    print(f"  滤波后最高 PSNR：{method_labels[best_psnr['method']]}，"
          f"{best_psnr['psnr_db']:.2f} dB，相对未滤波变化 "
          f"{best_psnr['psnr_db'] - baseline['psnr_db']:+.2f} dB。")
    print(f"  滤波后最高 SSIM：{method_labels[best_ssim['method']]}，"
          f"{best_ssim['ssim']:.4f}，相对未滤波变化 "
          f"{best_ssim['ssim'] - baseline['ssim']:+.4f}。")
    if noise == 'gaussian':
        print(f"  加噪后的平均误差：{baseline['mean_error']:+.2f}。")
        for record in filtered:
            print(f"  {method_labels[record['method']]}后平均误差：{record['mean_error']:+.2f}。")
print(f"高斯上限裁剪通道值比例：{observations['gaussian_upper_clipped_channel_fraction']:.2%}。")
print('输出尺寸及 uint8 类型保持：', summary['validation']['output_shape_preserved'])
print('依赖版本：', summary['versions'])
print('结果图片、指标和摘要已保存至：', output_dir)